# 07 · Evaluation

One prompt that works is not a test suite. Each evaluation case here carries its own pass/fail rubric, an LLM
judge grades the whole trace (not just the final answer), and a deterministic metric checks the safety rule.
See [docs/evaluation.md](../docs/evaluation.md) for the datasets, metrics and what the evaluations caught.

In [ ]:
from nbsetup import *

## A guardrail case

In [ ]:
case = json.loads((REPO / "growth-risk-agent/tests/eval/guardrails/guardrail-cases.json").read_text())["eval_cases"][1]
print(json.dumps(case, indent=2))

## A deterministic metric for the safety rule

`compliance_gate_respected` scores 0 for any write that wasn't both checked compliant *and* confirmed by a human.
No judge model involved.

In [ ]:
show("growth-risk-agent/tests/eval/compliance_gate.py", "def evaluate(instance):")

## Grade the guardrail cases

`run_eval.sh` generates traces by running each case against the agent code locally (with the deployed Compliance
Agent), adds the agent definitions the tool-use judge needs, and grades. About 2 minutes, and it calls Gemini.

In [ ]:
%%bash
cd "$REPO/growth-risk-agent"
tests/eval/run_eval.sh tests/eval/guardrails/guardrail-cases.json tests/eval/eval_config_guardrails.yaml 2>&1 \
  | grep -E "generate\]|_v[12]:|compliance_gate|mean_score|pass_rate"

The task cases (`tests/eval/datasets/task-cases.json` with `eval_config.yaml`) run the same way and take longer.
Keep the **held-out** sets in `tests/eval/holdout/` for a final check: grading them while you are still fixing the
agent turns them into training data.

The console also offers a managed **Agent Evaluation** with a simulated user. It has region and quota constraints
that are worth knowing first; see [docs/evaluation.md](../docs/evaluation.md#console-agent-evaluation).

Next: [08 · Model Armor](08_model_armor.ipynb).